# 1. Pull Raw Data

In [0]:
%sql
with 
    base as (
        select s.id as `ID`, 
               s.erp_cono as `Cono`, 
               s.erp_product_number as `Vallen ID`, 
               replace(i.descrip, ';', ' ') as `FullDesc`,
               s.category_l4_id
        from prd_bronze.stibo_ftp.sku s 
        inner join prd_bronze.openedge.icsp i 
        on s.erp_product_number = i.prod
        where s.st_row_current=1 
            and s.category_l4_id is not null 
            and i.st_row_current=1 
            and i.cono in (1,55) 
            and i.descrip is not null
    ), 

    hierarchy as (
        SELECT
            l1.id   AS `Level 1 ID`,
            l1.name AS `Level 1 Name`,
            l2.id   AS `Level 2 ID`,
            l2.name AS `Level 2 Name`,
            l3.id   AS `Level 3 ID`,
            l3.name AS `Level 3 Name`,
            l4.id   AS `Level 4 ID`,
            l4.name AS `Level 4 Name`
        FROM prd_bronze.stibo_ftp.hierarchy l1
        LEFT JOIN prd_bronze.stibo_ftp.hierarchy l2
            ON l2.parent_id = l1.id
        AND l2.level = 2
        AND l2.st_row_current = 1
        LEFT JOIN prd_bronze.stibo_ftp.hierarchy l3
            ON l3.parent_id = l2.id
        AND l3.level = 3
        AND l3.st_row_current = 1
        LEFT JOIN prd_bronze.stibo_ftp.hierarchy l4
            ON l4.parent_id = l3.id
        AND l4.level = 4
        AND l4.st_row_current = 1
        WHERE l1.level = 1
        AND l1.st_row_current = 1
    )

select *
from base b
inner join hierarchy h
on b.category_l4_id = h.`Level 4 ID` 
where h.`Level 4 ID` is not null

ID,Cono,Vallen ID,FullDesc,category_l4_id,Level 1 ID,Level 1 Name,Level 2 ID,Level 2 Name,Level 3 ID,Level 3 Name,Level 4 ID,Level 4 Name
I8IGTHF0H3HT,1,00892502046,D0748 F BLADE 7-1/4X48TH FERROVS,L4-12810826,L1-12810533,Tools & Equipment,L2-12810536,Power Tools,L3-12810584,Power Saw Blades & Accessories,L4-12810826,Circular Saw Blades
IF3JYMND0HXW,1,00892538579,50-101 ROUTER BIT,L4-12784351,L1-12783695,Cutting Tools,L2-12783697,Holemaking,L3-12783722,Specialty Drill Bits,L4-12784351,Router Drill Bits
I8HHYVD9KP78,1,008925DS0912BW5,9IN DEMO DEMON F/METAL,L4-12810833,L1-12810533,Tools & Equipment,L2-12810536,Power Tools,L3-12810584,Power Saw Blades & Accessories,L4-12810833,Reciprocating Saw Blades
I8HJI1MPX8DQ,1,01011981-34-34,81-34-34 MAGNIFIER 10X ILLUM CODDINGTON B&L,L4-12810909,L1-12810533,Tools & Equipment,L2-12810537,Test Instruments,L3-13340590,Inspection Equipment,L4-12810909,Magnifiers
IACF1JY0BD2N,1,01058203068,BLOCK GABLE 03068 1/4 CA BLE,L4-13322418,L1-12800755,Lifting & Rigging,L2-12800757,Chains & Ropes,L3-12800775,Ropes,L4-13322418,Ropes
S-8645626,1,15544235,66243506604 WHEEL 01 3 X 3/8 X 3/8 57A60-J8VB,L4-13482907,L1-12783670,Abrasives,L2-12783680,Sanding & Grinding,L3-12783682,Grinding & Cut-Off Abrasives,L4-13482907,Cut-Off Wheels
IF3JXQI90C35,1,01181200949,00949-1 PLUMBERS BRUSH 1IN SS,L4-12784258,L1-12783670,Abrasives,L2-12783671,Abrasives & Polishers,L3-12783674,Abrasive Brushes,L4-12784258,Tube Brushes
IF3J3TL50LJS,1,01181214610,14610 BRUSH TUBE BRASS LONG HANDLE 3/4X15OAL,L4-12784258,L1-12783670,Abrasives,L2-12783671,Abrasives & Polishers,L3-12783674,Abrasive Brushes,L4-12784258,Tube Brushes
I8HII08LSVIL,1,01181243407,BRUSH CONDENSER SS 7/8,L4-12784259,L1-12783670,Abrasives,L2-12783671,Abrasives & Polishers,L3-12783674,Abrasive Brushes,L4-12784259,Wire End Brushes
I8IGSDFTHXEL,1,01238200134,00134 4IN NRW CRMP WIRE WHEEL .0118 TLN-4,L4-12784260,L1-12783670,Abrasives,L2-12783671,Abrasives & Polishers,L3-12783674,Abrasive Brushes,L4-12784260,Wire Wheel Brushes


In [0]:
df = _sqldf.toPandas()
df.to_csv(r"../data/raw_data.txt", 
          index=False, 
          sep='|')

# 2. Clean Data

### Paths

In [0]:
import os
import sys

# This notebook sits in <repo>/notebooks/, so the package is one level up.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd

from product_classifier import PipelineConfig, data as data_mod, paths
from product_classifier.cleaning import (
    build_cleaner, clean_descriptions, load_cleaning_lists, upper,
    load_english_vocabulary,
)

print(paths.describe())

environment: Databricks
  data dir:   /Workspace/Users/patrick.white@vallen.com/Category Classification/NB Category Classifier/data
  output dir: /Workspace/Users/patrick.white@vallen.com/Category Classification/NB Category Classifier/outputs


### Data Parameters

In [0]:
cfg = PipelineConfig()

cfg.data.path = paths.data_path("raw_data.txt")
cfg.data.file_format = "csv"          # "csv" | "excel" | "delta" | "spark_table"
cfg.data.delimiter = "|"
cfg.data.text_col = "FullDesc"

CLEANED_PATH = paths.data_path("cleaned_data.txt")
print("reading :", cfg.data.path)
print("writing :", CLEANED_PATH)

reading : /Workspace/Users/patrick.white@vallen.com/Category Classification/NB Category Classifier/data/raw_data.txt
writing : /Workspace/Users/patrick.white@vallen.com/Category Classification/NB Category Classifier/data/cleaned_data.txt


### Cleaning Parameters

The identical recipe notebook 1 applies — the aggressive procedure in
`cleaning/descriptions.py`, not the simpler `CleaningConfig`. It has to
match: the vectorizer's vocabulary is fitted on this output, and a scoring
path that cleans differently sees mostly out-of-vocabulary words and fails
silently as a raised OOV rate rather than an error.

`build_cleaner` bundles the reference lists and the English word list into
one object, which gets saved with the model so notebook 5 cannot pick up a
different recipe.

In [0]:
!pip install nltk

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Must match notebook 01. The cleaner saved with the model carries this
# setting, so scoring expands exactly as training did -- but notebook 04
# cleans its own pull rather than reading notebook 01 output, so the
# toggle has to be repeated here.
EXPAND_ABBREVS = False

lists = {k: upper(v) for k, v in load_cleaning_lists().items()}
english_words = load_english_vocabulary()

cleaner = build_cleaner(lists=lists, vocabulary=english_words,
                        expand_abbrevs=EXPAND_ABBREVS)
print(cleaner.describe())

DescriptionCleaner(lists=['color_abbrevs', 'colors', 'sizes', 'unit_abbrevs', 'unit_plurals', 'units'], vocabulary=234,377 words, strip_leading_code=True, uppercase=False, expand_abbrevs=False)


### Load Data

In [0]:
df = data_mod.load_data(cfg.data)
print(f"{len(df):,} rows, {df.shape[1]} columns")

603,965 rows, 13 columns


In [0]:
to_pred = pd.read_csv(r"../data/to_predict.txt", 
                      dtype='str', 
                      delimiter='|')

df = df[~df['Vallen ID'].isin(to_pred['Vallen ID'])]

### Clean Data

In [0]:
cleaned = clean_descriptions(
    df, cfg.data.text_col,
    lists=lists,
    vocabulary=english_words,
    orig_col="FullDesc_orig",
    expand_abbrevs=EXPAND_ABBREVS,
    expansions=cleaner.expansions,
)

cleaned.to_csv(CLEANED_PATH, sep=cfg.data.delimiter, index=False)
print(f"\nwrote {len(cleaned):,} rows to {CLEANED_PATH}")
cleaned[["FullDesc_orig", cfg.data.text_col, "FirstWordReal"]].head(10)

[clean]   459,181  loaded
[clean]   459,181  after dropping null/blank descriptions
[clean]   459,181  after stripping 325,257 leading codes
[clean]   458,768  after the regex pass, blanks removed
[clean]       413  rows dropped in total (0.09%)

wrote 458,768 rows to /Workspace/Users/patrick.white@vallen.com/Category Classification/NB Category Classifier/data/cleaned_data.txt


,FullDesc_orig,FullDesc,FirstWordReal
0,D0748 F BLADE 7-1/4X48TH FERROVS,F BLADE TH FERROVS,0
1,50-101 ROUTER BIT,ROUTER BIT,0
2,9IN DEMO DEMON F/METAL,DEMO DEMON FMETAL,0
3,14610 BRUSH TUBE BRASS LONG HANDLE 3/4X15OAL,BRUSH TUBE BRASS LONG HANDLE OAL,0
4,BRUSH CONDENSER SS 7/8,BRUSH CONDENSER SS,1
5,00134 4IN NRW CRMP WIRE WHEEL .0118 TLN-4,4IN NRW CRMP WIRE WHEEL TLN4,0
6,07718 KIT REPLACE,KIT REPLACE,0
7,08209 12IN STD TWIST WIRE WHEEL .014 ST-12,12IN STD TWIST WIRE WHEEL ST12,0
8,10028 1-1/8 KNOT WIRE EN D BRUSH .020 EBB-41,KNOT WIRE EN D BRUSH EBB41,0
9,12256 5IN SINGLE RW WIRE CUP BRUSH (INTERNAL NUT,5IN SINGLE RW WIRE CUP BRUSH INTERNAL NUT,0


# 3. Train Model

In [0]:
from product_classifier.deploy import fit_final
from product_classifier.artifacts import (
    save_bundle, load_bundle, describe_bundle, attach_names,
    estimate_bundle_size,
)

In [0]:
# ---- what to predict ---------------------------------------------------
# IDs rather than names. joint_path matches labels by string equality, and
# a name can repeat under different parents, so IDs remove a real source
# of silent error. The names are kept for the lookup saved with the model.
ID_LEVELS = ["Level 1 ID", "Level 2 ID", "Level 3 ID", "Level 4 ID"]
NAME_LEVELS = ["Level 1 Name", "Level 2 Name", "Level 3 Name", "Level 4 Name"]

cfg.data.level_columns = ID_LEVELS
cfg.data.id_col = "Vallen ID"

cfg.model.n_levels = 4
cfg.model.nb.alpha = 0.001
cfg.model.vectorizer.max_features = 60_000


cfg.split.min_path_count = 20


cfg.model.vectorizer.ngram_range = (1, 2)
cfg.model.vectorizer.min_df = 1
cfg.model.vectorizer.max_df = 0.95
cfg.model.vectorizer.stop_words = "english"
cfg.model.vectorizer.kind = "tfidf"
cfg.model.vectorizer.sublinear_tf = True
cfg.model.vectorizer.norm = "l2"
cfg.model.vectorizer.use_idf = False
cfg.model.vectorizer.binary = True
# Fold each bigram to its sorted form, so "ball valve" and "valve ball"
# are one feature. Only bigrams; unigrams and trigrams keep their order.
cfg.model.vectorizer.unordered_bigrams = False
cfg.model.nb.fit_prior = True
cfg.model.prediction_mode = "joint_path"

cfg.clean_text = False                # cleaning already done above

levels = cfg.active_levels()
print("predicting:", levels)
print(f"alpha={cfg.model.nb.alpha}, max_features={cfg.model.vectorizer.max_features}, "
      f"kind={cfg.model.vectorizer.kind}, mode={cfg.model.prediction_mode}")

predicting: ['Level 1 ID', 'Level 2 ID', 'Level 3 ID', 'Level 4 ID']
alpha=0.001, max_features=60000, kind=tfidf, mode=joint_path


In [0]:
df = cleaned

print("null descriptions:", df[cfg.data.text_col].isna().sum())
for level in levels:
    print(f"{level}: {df[level].nunique():,} distinct, {df[level].isna().sum():,} null")
print("distinct full paths:", df[levels].drop_duplicates().shape[0])

null descriptions: 0
Level 1 ID: 23 distinct, 0 null
Level 2 ID: 158 distinct, 0 null
Level 3 ID: 791 distinct, 0 null
Level 4 ID: 4,652 distinct, 0 null
distinct full paths: 4652


### Train on everything

`fit_final` applies the same filtering a normal run would — drop null text
or labels, drop rare paths — and then trains on **all** of it, with no
holdout. That is right for a model being deployed and wrong for measuring
one: it has seen every row, so any accuracy figure computed on this data
would be meaningless. Use notebooks 6 and 7 for the measurements.

In [0]:
clf = fit_final(cfg, df=df)

[final] loaded 458,768 rows
[data] dropped 16,959 rows in paths seen < 20 times (4,652 -> 2,274 distinct paths)
[final] training on all 441,809 usable rows (no holdout)
[fit] vectorized 441,809 rows into 60,000 features (sparse, nnz=2,496,838) in 5.9s
[fit]   chunked fit: 441,809 rows in 23 chunk(s) of 20,000, 23 classes
[fit] trained 'Level 1 ID': 23 classes in 1.1s
[fit]   chunked fit: 441,809 rows in 23 chunk(s) of 20,000, 142 classes
[fit] trained 'Level 2 ID': 142 classes in 7.0s
[fit]   chunked fit: 441,809 rows in 23 chunk(s) of 20,000, 581 classes
[fit] trained 'Level 3 ID': 581 classes in 26.0s
[fit]   chunked fit: 441,809 rows in 23 chunk(s) of 20,000, 2,274 classes
[fit] trained 'Level 4 ID': 2,274 classes in 179.9s
[fit] 2,274 paths observed in training vs. 4,315,019,604 possible combinations -> scoring 0.000053% of the cross product


### Save

Four things have to survive, and one deliberately does not:

| Artifact | Why |
|---|---|
| `vectorizer.pkl` | Vocabulary and IDF weights — what maps a new description into the feature space the models were trained in |
| `model_Level_N_ID.pkl` | One Naive Bayes model per level |
| `path_index.pkl` | Which hierarchy paths were seen in training, and the label→column maps. `joint_path` cannot score without it |
| `config.pkl` / `config.json` | Reproducing the run |
| *the TF-IDF matrix* | **Not saved.** Derived data for the training rows; regenerating it from text takes seconds, and a stored copy can only fall out of step with the vectorizer that made it |

`classifier.pkl` is all of the above in one file. Prefer it for reloading —
the vectorizer and models cannot drift apart if they never separate. The
per-level files are there for inspecting or loading one component.

`manifest.json` records the shapes and library versions, and `load_bundle`
checks them: artifacts mixed from two training runs otherwise predict
confidently and wrongly.

### How big will it be?

`estimate_bundle_size` works from the array shapes, so it needs no disk
and can run before writing. Set `LIMIT_MB` to whatever your destination
allows and it reports the overage per file.

What dominates: `MultinomialNB` keeps **two** dense
`n_classes × n_features` float64 arrays per level — `feature_count_` and
`feature_log_prob_`. At 50,000 features, a Level 4 with 5,000 classes is
4 GB on its own.

In [0]:
LIMIT_MB = 500        # Databricks workspace files: 500. GitHub: 100.

est = estimate_bundle_size(clf, limit_mb=LIMIT_MB)
print(f"total {est.attrs['total_mb']:,.0f} MB | "
      f"largest file {est.attrs['largest_file_mb']:,.0f} MB | "
      f"over the limit by {est.attrs['largest_over_by_mb']:+,.0f} MB "
      f"({est.attrs['ratio_of_limit']:.1f}x)")
est

total 5,813 MB | largest file 2,906 MB | over the limit by +2,406 MB (5.8x)


,file,n_classes,bytes,mb,over_by_mb,fits
0,classifier.pkl,NaN,2906298528,2906.298528,2406.298528,False
1,model_Level_4_ID.pkl,2274.0,2183040000,2183.040000,1683.040000,False
2,model_Level_3_ID.pkl,581.0,557760000,557.760000,57.760000,False
3,model_Level_2_ID.pkl,142.0,136320000,136.320000,-363.680000,True
4,model_Level_1_ID.pkl,23.0,22080000,22.080000,-477.920000,True
5,vectorizer.pkl,NaN,6480000,6.480000,-493.520000,True
6,path_index.pkl,NaN,618528,0.618528,-499.381472,True


If that does not fit, the levers below are worth trying in order. The
first three cannot change a prediction; only `float32` can.

| Lever | Saves | Cost |
|---|---|---|
| `compress=3` | 2–4× on real log-probability arrays | Slower save/load |
| `include_combined=False` | ~half | No `classifier.pkl`; `load_bundle` falls back to the pieces automatically |
| `drop_fit_state=True` | ~half again | Saved models can no longer continue `partial_fit`. Prediction only needs `feature_log_prob_` |
| `float32=True` | ~half again | **Can** flip a prediction where two paths score within float32 resolution |

`include_combined=False` is close to free: `classifier.pkl` holds the same
data as the pieces, so writing both stores everything twice.

In [0]:
# Re-estimate with the reductions applied, before committing to them.
estimate_bundle_size(
    clf, limit_mb=LIMIT_MB,
    include_combined=False, drop_fit_state=True, float32=False,
)

,file,n_classes,bytes,mb,over_by_mb,fits
0,model_Level_4_ID.pkl,2274.0,1091520000,1091.520000,591.520000,False
1,model_Level_3_ID.pkl,581.0,278880000,278.880000,-221.120000,True
2,model_Level_2_ID.pkl,142.0,68160000,68.160000,-431.840000,True
3,model_Level_1_ID.pkl,23.0,11040000,11.040000,-488.960000,True
4,vectorizer.pkl,NaN,6480000,6.480000,-493.520000,True
5,path_index.pkl,NaN,618528,0.618528,-499.381472,True


In [0]:
MODEL_DIR = "../outputs/model_v1"

paths_written = save_bundle(
    clf, MODEL_DIR,
    n_training_rows=len(df),
    # Saved beside the model so predicted IDs can be turned back into
    # names later without re-querying the source.
    hierarchy_lookup=df[ID_LEVELS + NAME_LEVELS],
    # The recipe travels with the model, so scoring cleans identically.
    cleaning_cfg=cleaner,
    notes="joint_path, alpha=0.001, max_features=50000, tfidf, ID levels",
    # ---- size controls; see the table above --------------------------
    compress=3,
    include_combined=False,   # skip the duplicate all-in-one file
    drop_fit_state=True,      # keep only what prediction needs
    float32=False,            # last resort; can shift a prediction
)
for name, path in paths_written.items():
    print(f"{name:<24} {path}")

vectorizer               ../outputs/model_v1/vectorizer.pkl
config                   ../outputs/model_v1/config.pkl
path_index               ../outputs/model_v1/path_index.pkl
model::Level 1 ID        ../outputs/model_v1/model_Level_1_ID.pkl
model::Level 2 ID        ../outputs/model_v1/model_Level_2_ID.pkl
model::Level 3 ID        ../outputs/model_v1/model_Level_3_ID.pkl
model::Level 4 ID        ../outputs/model_v1/model_Level_4_ID.pkl
cleaning_config          ../outputs/model_v1/cleaning_config.pkl
hierarchy_lookup         ../outputs/model_v1/hierarchy_lookup.csv
manifest                 ../outputs/model_v1/manifest.json
config_json              ../outputs/model_v1/config.json


In [0]:
actual = describe_bundle(MODEL_DIR)
print(f"total on disk: {actual['size_mb'].sum():,.1f} MB | "
      f"largest file: {actual['size_mb'].max():,.1f} MB / {LIMIT_MB:,} MB limit")
actual

total on disk: 17.6 MB | largest file: 8.9 MB / 500 MB limit


,file,size_mb
0,cleaning_config.pkl,1.445645
1,config.json,0.000581
2,config.pkl,0.000379
3,hierarchy_lookup.csv,0.611457
4,manifest.json,0.001222
5,model_Level_1_ID.pkl,0.877461
6,model_Level_2_ID.pkl,1.563525
7,model_Level_3_ID.pkl,3.446964
8,model_Level_4_ID.pkl,8.931412
9,path_index.pkl,0.062040


### Verify the saved copy

Reload from the separate files rather than from `classifier.pkl`, so this
checks that the pieces genuinely agree, then confirm the reloaded model
predicts identically to the one still in memory. Worth doing once now
rather than discovering a bad save when it is needed.

In [0]:
reloaded = load_bundle(MODEL_DIR, from_pieces=True)

_sample = df[cfg.data.text_col].dropna().head(500)
_before = clf.predict(_sample)
_after = reloaded.predict(_sample)

_same_labels = (_before[levels].astype(str).values == _after[levels].astype(str).values).all()
_same_probs = ((_before["Probability"] - _after["Probability"]).abs() < 1e-12).all()
print(f"identical labels: {bool(_same_labels)} | identical probabilities: {bool(_same_probs)}")
assert _same_labels and _same_probs, "saved model does not reproduce the fitted one"

identical labels: True | identical probabilities: True


### Scoring new products

The production path. No labels anywhere — they were needed to fit the
model, not to use it. IDs come back from the model; `attach_names` maps
them to names using the lookup saved in the bundle.

In [0]:
new_texts = pd.Series([
    "brass ball valve 1 in npt threaded",
    "nitrile exam glove powder free blue",
])

pred = reloaded.predict(new_texts)
lookup = pd.read_csv(f"{MODEL_DIR}/hierarchy_lookup.csv")
attach_names(pred, lookup, ID_LEVELS, NAME_LEVELS)

,Level 1 ID,Level 2 ID,Level 3 ID,Level 4 ID,Probability,Level 1 Name,Level 2 Name,Level 3 Name,Level 4 Name
0,L1-12799358,L2-12799361,L3-12799407,L4-13531257,0.644266,"Pipes, Valves, Fittings",Valves,Bronze Valves & Actuators,Bronze Ball Valves
1,L1-12782908,L2-12782914,L3-12782962,L4-12783317,0.998412,Safety & Security,Personal Protective Equipment (PPE),Gloves,Disposable Gloves
